In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV

In [6]:
# Load data
df = pd.read_csv('data/tournament_model_ml.csv')

In [7]:
# Features and target
features = ['3P%_diff', 'AST_diff', 'FG%_diff', 'FT%_diff', 'SRS_diff',
            'TOV_diff', 'TRB_diff', 'seed_diff', 'win_pct_diff']

            # Train/test split by season
train = df[df['season'] <= 2022]
test  = df[df['season'] >= 2023]

X_train, y_train = train[features], train['win_label']
X_test,  y_test  = test[features],  test['win_label']

print(f"Train rows: {len(train)} | Test rows: {len(test)}")

Train rows: 754 | Test rows: 314


In [8]:
# Parameter grid to search over
param_dist = {
    'n_estimators':      [100, 200, 300, 500, 750],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
    'bootstrap':         [True, False]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)


# Randomized search — tries 50 random combinations, 5-fold CV
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,           # number of random combos to try
    cv=5,                # 5-fold cross-validation
    scoring='roc_auc',   # optimize for AUC
    random_state=42,
    n_jobs=-1,
    verbose=1
)


In [9]:
search.fit(X_train, y_train)

print(f"\nBest Parameters: {search.best_params_}")
print(f"Best CV AUC:     {search.best_score_:.4f}")

# Evaluate best model on test set
best_rf = search.best_estimator_

y_pred  = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

# Feature importances from best model
importances = sorted(zip(features, best_rf.feature_importances_),
                     key=lambda x: x[1], reverse=True)
print("\nFeature Importances:")
for feat, imp in importances:
    print(f"  {feat:<15} {imp:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Parameters: {'n_estimators': 750, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 5, 'bootstrap': True}
Best CV AUC:     0.8233

Test Accuracy:  0.7707
Test ROC-AUC:   0.8445

Confusion Matrix:
[[121  36]
 [ 36 121]]

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.77      0.77       157
           1       0.77      0.77      0.77       157

    accuracy                           0.77       314
   macro avg       0.77      0.77      0.77       314
weighted avg       0.77      0.77      0.77       314


Feature Importances:
  SRS_diff        0.3478
  TRB_diff        0.1643
  seed_diff       0.1502
  AST_diff        0.1199
  win_pct_diff    0.0750
  FT%_diff        0.0460
  FG%_diff        0.0353
  TOV_diff        0.0319
  3P%_diff        0.0296
